In [176]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from dotenv.ipython import load_dotenv
from IPython.display import display, Markdown

In [177]:
load_dotenv(override=True)

True

In [178]:
model = ChatOpenAI(model="gpt-4o", temperature=0)

In [179]:
from langgraph.checkpoint.memory import InMemorySaver

In [180]:

agent = create_agent(
    model=model,
    system_prompt="You are a helpful assistant that helps answer questions about the world.",)

In [181]:
resp = agent.invoke(input={"messages": [{"role": "user", "content": "Je mappele Saifeddine"}]})

In [182]:
print(resp['messages'] [-1].content)

Bonjour Saifeddine ! Comment puis-je vous aider aujourd'hui ?


In [183]:
resp = agent.invoke(input={"messages": [{"role": "user", "content": "comment je mappele"}]})

In [184]:
print(resp['messages'] [-1].content)

Je suis désolé, mais je ne connais pas votre nom. Si vous me le dites, je pourrais m'en souvenir pour cette conversation.


In [185]:
from langchain.agents.middleware import ModelRequest , ModelResponse, wrap_model_call
from langchain.messages import  SystemMessage , HumanMessage , AIMessage

In [186]:
basic_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
advanced_llm = ChatOpenAI(model="gpt-4o", temperature=0)

In [187]:
@wrap_model_call
def dynamic_model_selection(request : ModelRequest , handler)->ModelResponse:
  env = request.runtime.context.get("env","test")
  if(env == "prod") :
    model = advanced_llm
    print("Using advanced llm")
  else :
    model = basic_llm
    print("Using basic llm")
  return handler(request.override(model=model))


In [188]:
agent2 = create_agent(
    model =basic_llm,
    tools = [],
    middleware=[dynamic_model_selection],
    debug=True
    )

In [189]:
resp=agent2.invoke(input={"messages":[HumanMessage("C quoi un agent AI")]},context={"env":"prod"}
    )

[values] {'messages': [HumanMessage(content='C quoi un agent AI', additional_kwargs={}, response_metadata={}, id='e5012542-487c-4430-a8ab-9768e25ccf80')]}
Using advanced llm
[updates] {'model': {'messages': [AIMessage(content="Un agent AI (agent d'intelligence artificielle) est un programme informatique conçu pour effectuer des tâches spécifiques de manière autonome ou semi-autonome en utilisant des techniques d'intelligence artificielle. Ces agents peuvent percevoir leur environnement, prendre des décisions basées sur des algorithmes et des modèles d'apprentissage, et agir en conséquence pour atteindre des objectifs prédéfinis. \n\nLes agents AI peuvent être utilisés dans divers domaines, tels que :\n\n1. **Assistants virtuels** : Comme Siri, Alexa ou Google Assistant, qui aident les utilisateurs à effectuer des tâches quotidiennes en répondant à des questions, en envoyant des messages, ou en contrôlant des appareils connectés.\n\n2. **Agents conversationnels (chatbots)** : Utilisés d

In [190]:
print(resp['messages'] [-1].content)

Un agent AI (agent d'intelligence artificielle) est un programme informatique conçu pour effectuer des tâches spécifiques de manière autonome ou semi-autonome en utilisant des techniques d'intelligence artificielle. Ces agents peuvent percevoir leur environnement, prendre des décisions basées sur des algorithmes et des modèles d'apprentissage, et agir en conséquence pour atteindre des objectifs prédéfinis. 

Les agents AI peuvent être utilisés dans divers domaines, tels que :

1. **Assistants virtuels** : Comme Siri, Alexa ou Google Assistant, qui aident les utilisateurs à effectuer des tâches quotidiennes en répondant à des questions, en envoyant des messages, ou en contrôlant des appareils connectés.

2. **Agents conversationnels (chatbots)** : Utilisés dans le service client pour interagir avec les utilisateurs, répondre à leurs questions et résoudre des problèmes courants.

3. **Systèmes de recommandation** : Qui suggèrent des produits, des films ou de la musique en fonction des pr

In [191]:
rep=agent.invoke(input={"messages":[HumanMessage("je mappele mohamed")]})

In [192]:
print(rep['messages'] [-1].content)

Bonjour Mohamed ! Comment puis-je vous aider aujourd'hui ?


In [193]:
from langgraph.checkpoint.memory import InMemorySaver
memory=InMemorySaver()
agent = create_agent(
    model=model,
    system_prompt="You are a helpful assistant that helps answer questions about the world.",
    checkpointer=memory
)

In [194]:
config = {"configurable": {"thread_id": "1"}}
rep=agent.invoke(input={"messages":[HumanMessage("je mappele mohamed")]}, 
                 config=config)

In [195]:
print(rep['messages'] [-1].content)

Bonjour Mohamed ! Comment puis-je vous aider aujourd'hui ?


In [196]:
rep=agent.invoke(input={"messages":[HumanMessage("comment je mappele ")]}, 
                 config=config)

In [197]:
print(rep['messages'] [-1].content)

Vous avez dit que vous vous appelez Mohamed. Est-ce correct ?


In [198]:
from langchain.tools import tool

In [199]:
@tool
def get_weather(city:str):
    """ 
    Get The weather of the given city 
    """
    return f"la meteo a {city} est ensoleillé"
@tool
def get_employee_info(employee_name:str):
    """
    Get information about an employee given their name
    """
    return f"Information about employee {employee_name}: Age: 30, Position: Software Engineer, Department: IT"

In [200]:
agent4 = create_agent(
    model="openai:gpt-4o",
    tools=[get_weather,get_employee_info],
    system_prompt="answer the question using only  provided tools",
    checkpointer=memory
)

In [201]:
resp=agent4.invoke(input={"messages":[HumanMessage("quel est la meteo a paris ")]},config=config)

In [202]:
print(resp['messages'] [-1].content)

La météo à Paris est ensoleillée. Profitez du beau temps !


In [203]:
load_dotenv(override=True)

True

In [204]:
from langchain_tavily import TavilySearch 

In [205]:
tavily=TavilySearch(max_results=10,search_depth="advanced")

In [206]:
@tool
def search_web(query:str,max_results=10):
    """ Search for the best valorant players in the world"""
    print(f"search_web invoked with {query}")
    results = tavily.invoke({"query":query})
    return results
    

In [207]:
agent5 = create_agent(
    model="openai:gpt-4o",
    tools=[get_weather,get_employee_info, search_web],
    system_prompt="answer the question using only  provided tools",
    checkpointer=memory
)

In [208]:
resp=agent5.invoke(input={"messages":[HumanMessage("qui sont les meilleurs joueurs de valorant ")]},config=config)  

search_web invoked with best Valorant players in the world


In [209]:
print(resp['messages'] [-1].content)

Voici quelques-uns des meilleurs joueurs de Valorant dans le monde :

1. **aspas** - Connu pour son style de jeu agressif et précis, il est considéré comme l'un des meilleurs joueurs de l'histoire de Valorant.

2. **TenZ** - Un joueur très célèbre dans le monde de Valorant, reconnu pour ses compétences mécaniques exceptionnelles.

3. **صالحة** - Connu pour ses performances impressionnantes et son influence dans l'esport Valorant.

4. **Boaster** - Réputé non seulement pour ses compétences individuelles mais aussi pour ses capacités de leader.

5. **tarik** - Ancien champion du monde et streamer de Valorant très populaire.

Ces joueurs se sont distingués par leur talent, leur stratégie et leur contribution à la scène esport de Valorant.


In [213]:
from langchain_experimental.tools import PythonREPLTool

In [218]:
repl_tool=PythonREPLTool(sanitize_input=False)

In [221]:
agent6=create_agent(
    model="openai:gpt-4o",
    tools=[repl_tool],
    system_prompt="""generer et executer le code python en plancant le code python et le resultat dans le fichier doc.txt""",
    debug=True)

In [222]:
resp=agent6.invoke(input={"messages":[HumanMessage("ecrire un code python qui affiche les 10 premiers nombres premiers ")]})

[values] {'messages': [HumanMessage(content='ecrire un code python qui affiche les 10 premiers nombres premiers ', additional_kwargs={}, response_metadata={}, id='d7736718-3d0f-484a-b82c-4b4a637769e7')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 117, 'prompt_tokens': 116, 'total_tokens': 233, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_11da807510', 'id': 'chatcmpl-DR85R5QP3omdbsM2M98UUPteynC9n', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d5b99-fe0f-7f43-965b-2ce8a7024f71-0', tool_calls=[{'name': 'Python_REPL', 'args': {'query': 'def premiers_nombres(n):\n    nombres_premiers = []\n    n